In [ ]:
# HW24: Обучение модели для диабета и интеграция с MLflow

import os
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Настройка MLflow (при необходимости поменяйте URI на свой)
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("diabets_experiment")

ROOT = Path("..").resolve()
MODEL_EXPORT_DIR = ROOT / "mlapp" / "model"
MODEL_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Model export dir:", MODEL_EXPORT_DIR)


In [ ]:
# Загрузка датасета диабета
# В условии задано load_diabets(scaled=False), но в sklearn используется
# функция load_diabetes. Здесь используем её, а масштабирование добавим
# через StandardScaler в пайплайне.

raw = load_diabetes()
X = pd.DataFrame(raw.data, columns=raw.feature_names)
y = pd.Series(raw.target, name="target")

print("Фичи:", list(X.columns))
print("Размер выборки:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
# Обучение RandomForestRegressor + StandardScaler и логирование в MLflow

n_estimators = 200
max_depth = 6
random_state = 42

pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "rf",
            RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                random_state=random_state,
                n_jobs=-1,
            ),
        ),
    ]
)

with mlflow.start_run(run_name="diabets_rf_run") as run:
    mlflow.log_params({
        "model_type": "RandomForestRegressor",
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "random_state": random_state,
    })

    pipeline.fit(X_train, y_train)

    train_score = pipeline.score(X_train, y_train)
    test_score = pipeline.score(X_test, y_test)

    mlflow.log_metrics({
        "r2_train": float(train_score),
        "r2_test": float(test_score),
    })

    # Логируем модель и регистрируем её под именем "diabets"
    registered_model_name = "diabets"

    model_info = mlflow.sklearn.log_model(
        sk_model=pipeline,
        artifact_path="model",
        registered_model_name=registered_model_name,
    )

    print("Run ID:", run.info.run_id)
    print("Registered model name:", registered_model_name)
    print("Model version:", model_info.registered_model_version)

    model_version = model_info.registered_model_version


In [ ]:
# Загрузка зарегистрированной модели по имени и версии

assert "model_version" in globals(), "Сначала выполните предыдущую ячейку, чтобы получить model_version"

model_uri = f"models:/diabets/{model_version}"
print("Model URI:", model_uri)

loaded_model = mlflow.sklearn.load_model(model_uri)

sample = X_test.iloc[[0]]
print("Sample features:\n", sample)
print("Prediction:", loaded_model.predict(sample)[0])


In [ ]:
# Экспорт модели для использования в FastAPI сервисе

# Сохраняем модель, загруженную из MLflow реестра, в директорию mlapp/model

mlflow.sklearn.save_model(sk_model=loaded_model, path=str(MODEL_EXPORT_DIR))

print("Model saved to:", MODEL_EXPORT_DIR)
print("Теперь можно запускать FastAPI сервис (локально или через Docker),")
print("и ручка /api/v1/predict будет использовать эту модель.")
